# Lab 3: Early Fusion + MLP Classification (MSCOCO)

In [2]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# Load features
image_features = np.load("coco_features/image_features.npy")
caption_features = np.load("coco_features/caption_features.npy")
labels = np.load("coco_features/labels.npy")

# Early fusion
X = np.concatenate([image_features, caption_features], axis=1)  # (5000, 2816)
y = labels  # (5000, 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim=2816, hidden_dim=512, output_dim=80):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()  # multilabel
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 100
batch_size = 64

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))
    total_loss = 0
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        xb, yb = X_train[idx].to(device), y_train[idx].to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test.to(device))
    test_loss = criterion(preds, y_test.to(device)).item()
print("Final Test Loss:", test_loss)


X shape: (4952, 2816)
y shape: (4952, 80)
Epoch 1/100, Loss: 8.2576
Epoch 2/100, Loss: 4.7086
Epoch 3/100, Loss: 4.0523
Epoch 4/100, Loss: 3.6631
Epoch 5/100, Loss: 3.4136
Epoch 6/100, Loss: 3.2400
Epoch 7/100, Loss: 3.1043
Epoch 8/100, Loss: 2.9361
Epoch 9/100, Loss: 2.8277
Epoch 10/100, Loss: 2.6921
Epoch 11/100, Loss: 2.5830
Epoch 12/100, Loss: 2.4758
Epoch 13/100, Loss: 2.3380
Epoch 14/100, Loss: 2.2714
Epoch 15/100, Loss: 2.1646
Epoch 16/100, Loss: 2.1319
Epoch 17/100, Loss: 1.9975
Epoch 18/100, Loss: 1.9261
Epoch 19/100, Loss: 1.8506
Epoch 20/100, Loss: 1.7406
Epoch 21/100, Loss: 1.6507
Epoch 22/100, Loss: 1.5876
Epoch 23/100, Loss: 1.4800
Epoch 24/100, Loss: 1.4306
Epoch 25/100, Loss: 1.3751
Epoch 26/100, Loss: 1.2601
Epoch 27/100, Loss: 1.2223
Epoch 28/100, Loss: 1.1720
Epoch 29/100, Loss: 1.0721
Epoch 30/100, Loss: 1.0544
Epoch 31/100, Loss: 0.9724
Epoch 32/100, Loss: 0.9043
Epoch 33/100, Loss: 0.8484
Epoch 34/100, Loss: 0.8012
Epoch 35/100, Loss: 0.7532
Epoch 36/100, Loss: 0.